# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/neha-raniii/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [10]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/neha-raniii/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

import pandas as pd
import numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

print(f"{len(df):,} rows loaded")

30,000 rows loaded


In [11]:
# Recreate Week-4 baseline: CTR-below-position-norm rule
tier_avg_ctr = df.groupby('position_tier')['ctr'].transform('mean')
df['tier_avg_ctr'] = tier_avg_ctr
df['baseline_score'] = np.where(
    (df['avg_position'] > 0) & (df['impressions_90d'] >= 500) & (df['tier_avg_ctr'] > df['ctr']),
    (df['tier_avg_ctr'] - df['ctr']) * df['impressions_90d'],
    0
)

y = df['is_declining_label'].values
baseline_p50 = precision_at_k(df['baseline_score'], y, 50)
print(f"Week-4 baseline Precision@50: {baseline_p50:.3f}")


Week-4 baseline Precision@50: 0.440


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

Method choice: Random Forest Classifier. My lane (Refresh / Content Opportunity Scoring) is a scoring/ranking task, and a random forest handles multiple weak, possibly-interacting signals (impressions, position, CTR, word count, freshness) better than a single linear rule or a shallow tree - which matters here because Week-4's signal audit already showed one intuitive signal (staleness) pointing the wrong way, meaning the real pattern is not obvious from any single feature. This also matches the reference result shown throughout the starter pipeline, where random forest was the strongest performer (Precision@50 = 0.740) against the same kind of baseline rule I built in Week 4.


In [12]:
# Model built and compared in Section 3.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

Split design: Client-holdout (grouped) split - entire clients kept out of training, so the model is tested only on clients it never saw. A plain random row split would let pages from the same client appear in both train and test, letting the model partly memorize client-specific patterns rather than learn a signal that generalizes to a brand-new client. This matches the validation approach used in the starter pipeline and in my own Week-3 exploration, where a client-wise split changed the result meaningfully compared to a random split.

In [13]:
from sklearn.model_selection import GroupShuffleSplit

feature_cols = ['impressions_90d', 'avg_position', 'ctr', 'word_count',
                 'days_since_last_update', 'search_volume', 'engagement_rate']
model_df = df.dropna(subset=feature_cols + ['client_id']).copy()

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(model_df, groups=model_df['client_id']))

train_df = model_df.iloc[train_idx]
test_df = model_df.iloc[test_idx]

print(f"Train: {len(train_df):,} rows, {train_df['client_id'].nunique()} clients")
print(f"Test:  {len(test_df):,} rows, {test_df['client_id'].nunique()} clients")
print(f"Overlap check - shared clients: {len(set(train_df['client_id']) & set(test_df['client_id']))}")


Train: 14,160 rows, 21 clients
Test:  5,858 rows, 8 clients
Overlap check - shared clients: 0


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

Training a Random Forest on the client-holdout split, then comparing Precision@50 against the Week-4 baseline rule on the same test set.


In [14]:
from sklearn.ensemble import RandomForestClassifier

X_train, y_train = train_df[feature_cols], train_df['is_declining_label']
X_test, y_test = test_df[feature_cols], test_df['is_declining_label']

model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

model_scores = model.predict_proba(X_test)[:, 1]
baseline_scores_test = test_df['baseline_score'].values

model_p50 = precision_at_k(model_scores, y_test.values, 50)
baseline_p50_test = precision_at_k(baseline_scores_test, y_test.values, 50)

comparison = pd.DataFrame({
    'Method': ['Week-4 baseline rule', 'Random Forest (this week)'],
    'Precision@50': [baseline_p50_test, model_p50]
})
print(comparison.to_string(index=False))


                   Method  Precision@50
     Week-4 baseline rule          0.56
Random Forest (this week)          0.74


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

Result: The Random Forest (Precision@50 = 0.740) clearly beats the Week-4 baseline rule on the same client-holdout test set (Precision@50 = 0.560) - a meaningful jump from 28 to 37 correct pages in the top 50. This confirms the Week-4 signal audit's finding that no single hand-written rule captures the full pattern; the model finds a stronger combination of signals than the CTR-vs-position rule alone.

In [15]:
importances = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)
print(importances.to_string(index=False))


               feature  importance
       impressions_90d    0.260505
          avg_position    0.219545
            word_count    0.212176
                   ctr    0.113456
       engagement_rate    0.069666
         search_volume    0.069201
days_since_last_update    0.055451


Error and feature interpretation: The model leans most on impressions_90d (26%), avg_position (22%), and word_count (21%) - together explaining nearly 70% of its decisions. Notably, days_since_last_update (staleness) is the LEAST important feature (5.5%), which lines up with the Week-4 finding that staleness alone was an unreliable signal on this data. This suggests the model is not just recombining my hand-written rule (which used ctr and avg_position) - it found word_count and impressions_90d matter more than I assumed, which a fixed rule would have missed entirely.

Where the model likely struggles: pages with very low impressions_90d (thin data) probably get less reliable scores, since the model has fewer examples to learn from at low volume - the baseline rule also required impressions_90d >= 500 for exactly this reason, and the model has no equivalent explicit safeguard. A next step would be checking Precision@50 separately for high-volume vs low-volume pages to see if the model's advantage holds at both ends.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.